# 05 -- Fetch Staff Subscriptions

Staff-migration variant of `05_Fetch_Subscriptions.ipynb`. Loads the exact
active-subscription snapshot `04_Create_Staff_Accounts.ipynb` already pulled
(`staff_subscriptions_active_raw` -- not re-queried here, so this can't drift
from the set that decided which accounts got created), then for each one:

1. Resolves `TargetAccountNumber` -- a straight join of the subscription's
   `AccountCode` against `04`'s `GeneratedAccountNumber` results
   (`load_staff_account_number_map()`). No bucket-account routing, no
   `Reference`-based special cases -- every staff subscription goes onto its
   own real account, or nowhere at all if that account isn't ready.
2. Looks up its real address + radius username from Voyager
   (`get_voyager_address`), exactly as `05_Fetch_Subscriptions.ipynb` does.

Output feeds `06_Create_Staff_Addresses.ipynb` and
`07_Create_Staff_Subscription_Orders.ipynb`.

## 1. Setup

In [7]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from concurrent.futures import ThreadPoolExecutor, as_completed

logger = get_logger("fetch_staff_subscriptions")

TEST_ROW_LIMIT = None  # Testing limiter -- set to None for a full run.


In [8]:
# HARDCODED_TOKEN = "c5bf2481-f3fd-498b-a260-27c6e783782f"

# token_manager._token = HARDCODED_TOKEN
# token_manager._expires_at = datetime.now() + timedelta(hours=1)  # adjust to match the real token's actual TTL

## 2. Load the active-subscriptions snapshot

Same rows `04_Create_Staff_Accounts.ipynb` used to decide which accounts to
create -- loaded from disk, not re-queried, so the two notebooks can't see
different data if the underlying MySQL table changes in between runs.

In [9]:
df_subscriptions = load_df("staff_subscriptions_active_raw")
logger.info(f"Loaded {len(df_subscriptions):,} active staff subscriptions from 04's snapshot")

if TEST_ROW_LIMIT is not None:
    df_subscriptions = df_subscriptions.head(TEST_ROW_LIMIT)  # Testing limiter -- remove/raise for a full run.
    logger.info(f"TEST_ROW_LIMIT active -- trimmed to {len(df_subscriptions):,} rows")

df_subscriptions.head()


2026-07-31 13:53:30,781 [INFO] Loaded 230 active staff subscriptions from 04's snapshot


,_rowid,_rowmodified,_sourceid,_DataSource,AccountCode,ServiceType,SubscriptionUSN,SubscriptionLabel,SubscriptionStartDate,SubscriptionEndDate,...,CircuitType,Server,CustomerSuppliedReference,_notforreports_VoyagerOrderHistory,_notforreports_LegacyServiceDescription,NextPlanCode,NextPlanStartDate,NextQuantity,NextCustomPrice,SalesAgentCode
0,3044743789,2024-04-19 11:24:20,248267,vBill,99972482,Broadband - Fibre,V112961768,ryan.beaumont@vygr.net,2024-04-19,NaN,...,UFB Max/500,NaN,NaN,PROV-200895,NaN,NaN,NaN,NaN,NaN,NaN
1,17859,2025-09-02 01:01:11,4915,vBill,99999854,Promotional Credit,V111063442,V111063442,2016-06-01,NaN,...,NaN,NaN,NaN,Contract: Staff FOV UFB [KCC-190-93580],Grant Schneider,NaN,NaN,NaN,NaN,NaN
2,2929352075,2026-07-03 20:30:30,247449,vBill,99999651,Broadband - Fibre,V112955638,mark.mackay@vygr.net,2024-03-14,NaN,...,UFB Max/500,NaN,MACKAY_UFB920,PROV-199690,NaN,NaN,NaN,NaN,NaN,NaN
3,1743067968,2025-09-02 01:01:34,235413,vBill,45810287,IP Voice Addon,V112852694,22c30d2e-4720-204e-2504-616430960914 (DDIs),2023-03-22,NaN,...,NaN,NaN,NaN,2024-07-12: Remove 6494444444; 2024-06-13: Add...,NaN,NaN,NaN,NaN,NaN,NaN
4,288034,2019-05-07 18:06:59,116073,vBill,45810287,Hardware Rental,V111975470,J3N7S18518921053,2019-01-22,NaN,...,NaN,NaN,NaN,PROV-79104,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Resolve target account for every subscription

Direct join on `AccountCode` -- `load_staff_account_number_map()` returns
`{AccountCode: GeneratedAccountNumber}` for every staff account that both
(a) was actually created (`status == "created"`, not `"exists"` -- see the
idempotency note in `04_Create_Staff_Accounts.ipynb`) and (b) had its
OneBill-assigned account number successfully extracted from the create
response.

Any subscription whose account isn't in that map (creation failed, or
`GeneratedAccountNumber` couldn't be extracted) gets `TargetAccountNumber =
NaN` here -- there's no bucket-account fallback for staff, so those rows are
flagged and excluded rather than silently rerouted. Check the "unresolved"
table below before continuing to `06_Create_Staff_Addresses.ipynb`.

In [10]:
own_account_map = load_staff_account_number_map()  # {AccountCode: GeneratedAccountNumber}
logger.info(f"{len(own_account_map):,} staff accounts have a confirmed OneBill accountNumber from 04's results")

df_subscriptions["AccountCode"] = df_subscriptions["AccountCode"].astype(str)
df_subscriptions["TargetAccountKey"] = "own_account"
df_subscriptions["TargetAccountNumber"] = df_subscriptions["AccountCode"].map(own_account_map)

unresolved = df_subscriptions[df_subscriptions["TargetAccountNumber"].isna()]
if not unresolved.empty:
    logger.warning(
        f"{len(unresolved):,} / {len(df_subscriptions):,} subscriptions have no resolved TargetAccountNumber "
        f"-- their account either failed to create or its GeneratedAccountNumber couldn't be extracted "
        f"(see 04_Create_Staff_Accounts.ipynb's failure / missing-number summaries). Dropping them here; "
        f"re-run 04 for these AccountCodes and come back to this notebook once they resolve."
    )

df_subscriptions = df_subscriptions[df_subscriptions["TargetAccountNumber"].notna()].copy()
logger.info(f"{len(df_subscriptions):,} subscriptions have a resolved TargetAccountNumber and will proceed")

unresolved[["SubscriptionUSN", "AccountCode"]].head(20) if not unresolved.empty else unresolved


2026-07-31 13:53:31,889 [INFO] 64 staff accounts have a confirmed OneBill accountNumber from 04's results
2026-07-31 13:53:31,908 [WARNING] 20 / 230 subscriptions have no resolved TargetAccountNumber -- their account either failed to create or its GeneratedAccountNumber couldn't be extracted (see 04_Create_Staff_Accounts.ipynb's failure / missing-number summaries). Dropping them here; re-run 04 for these AccountCodes and come back to this notebook once they resolve.
2026-07-31 13:53:31,916 [INFO] 210 subscriptions have a resolved TargetAccountNumber and will proceed


,SubscriptionUSN,AccountCode
2,V112955638,99999651
9,V111063772,99999945
17,V111055927,99999734
20,V111063236,99999504
28,V112704440,99999945
33,V112942511,99999504
40,V112996160,99999651
53,V112852447,99999651
65,V111063350,99999651
85,V112896964,99999651


## 4. Voyager address lookup (circuits + address-search)

Identical to `05_Fetch_Subscriptions.ipynb` step 4 -- `SupplierServiceID` ->
`GET .../fibre/v1/circuits/{id}` (gives `radiusUsers[0]` + `locationId`) ->
`GET .../address-search/v3/addresses/id/{locationId}` (gives the actual
street address, city, postcode, region). Region name mapped to a 3-char ISO
code via `NZ_Regions.xlsx`.

In [11]:
if VOYAGER_CCP_KEY is None or VOYAGER_PARTNER_ID is None:
    logger.warning("VOYAGER_CCP_KEY / VOYAGER_PARTNER_ID not set -- every Voyager lookup below will fail. Set them in .env.")

blank_supplier_ids = df_subscriptions["SupplierServiceID"].isna() | (df_subscriptions["SupplierServiceID"].astype(str).str.strip() == "")
if blank_supplier_ids.all():
    logger.warning(
        "SupplierServiceID is blank for EVERY subscription in this batch -- the Voyager circuits lookup "
        "can't run at all without it, so every ParsedAddress_* field below will be None. Check the MySQL "
        "source data / STAFF_ACTIVE_SUBSCRIPTIONS_QUERY in 04_Create_Staff_Accounts.ipynb before re-running."
    )
elif blank_supplier_ids.any():
    logger.warning(f"{blank_supplier_ids.sum():,} / {len(df_subscriptions):,} subscriptions have a blank SupplierServiceID")

voyager_session = new_voyager_session(max_workers=MAX_WORKERS)


def _lookup_row(supplier_service_id):
    return get_voyager_address(voyager_session, supplier_service_id)


logger.info(f"Looking up Voyager address for {len(df_subscriptions):,} subscriptions with {MAX_WORKERS} workers...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    voyager_results = list(executor.map(_lookup_row, df_subscriptions["SupplierServiceID"]))

existing_parsed_cols = [c for c in df_subscriptions.columns if c.startswith("ParsedAddress_")]
if existing_parsed_cols:
    df_subscriptions = df_subscriptions.drop(columns=existing_parsed_cols)  # safe to re-run this cell

address_parts = pd.DataFrame(voyager_results, index=df_subscriptions.index)
address_parts = address_parts.add_prefix("ParsedAddress_")
df_subscriptions = pd.concat([df_subscriptions, address_parts], axis=1)

unparsed = df_subscriptions[~df_subscriptions["ParsedAddress_parsed_ok"]]
if not unparsed.empty:
    logger.warning(f"{len(unparsed):,} subscriptions could not be resolved to a Voyager address")
    error_summary = (
        unparsed["ParsedAddress_error"].value_counts(dropna=False)
        .rename_axis("error").reset_index(name="count")
    )
    logger.info("Error breakdown:\n" + error_summary.to_string(index=False))

df_subscriptions[[
    "SubscriptionLabel", "SupplierServiceID",
    "ParsedAddress_addLine1", "ParsedAddress_addLine2", "ParsedAddress_city",
    "ParsedAddress_postcode", "ParsedAddress_region_iso", "ParsedAddress_region_code_raw", "ParsedAddress_radius_user",
    "ParsedAddress_parsed_ok", "ParsedAddress_error",
]].head(20)


2026-07-31 13:53:33,531 [WARNING] 143 / 210 subscriptions have a blank SupplierServiceID


2026-07-31 13:53:33,636 [INFO] Looking up Voyager address for 210 subscriptions with 3 workers...
2026-07-31 13:54:06,322 [WARNING] Voyager 429 on https://api.voyager.nz/fibre/v1/circuits/1639381286 — retry 1/6 in 60.2s
2026-07-31 13:54:06,821 [WARNING] Voyager 429 on https://api.voyager.nz/fibre/v1/circuits/ENVOYB02122402 — retry 1/6 in 60.3s
2026-07-31 13:54:07,320 [WARNING] Voyager 429 on https://api.voyager.nz/fibre/v1/circuits/ENVOYB02308809 — retry 1/6 in 59.5s
2026-07-31 13:55:10,344 [WARNING] 210 subscriptions could not be resolved to a Voyager address
2026-07-31 13:55:10,354 [INFO] Error breakdown:
                                                                                                               error  count
                                                                           no SupplierServiceID on this subscription    143
circuits lookup failed: 404 Client Error: Not Found for url: https://api.voyager.nz/fibre/v1/circuits/ENVOYB02528321      1
    circuits 

,SubscriptionLabel,SupplierServiceID,ParsedAddress_addLine1,ParsedAddress_addLine2,ParsedAddress_city,ParsedAddress_postcode,ParsedAddress_region_iso,ParsedAddress_region_code_raw,ParsedAddress_radius_user,ParsedAddress_parsed_ok,ParsedAddress_error
0,ryan.beaumont@vygr.net,ENVOYB02528321,None,None,None,None,None,None,None,False,circuits lookup failed: 404 Client Error: Not ...
1,V111063442,NaN,None,None,None,None,None,None,None,False,no SupplierServiceID on this subscription
3,22c30d2e-4720-204e-2504-616430960914 (DDIs),NaN,None,None,None,None,None,None,None,False,no SupplierServiceID on this subscription
4,J3N7S18518921053,NaN,None,None,None,None,None,None,None,False,no SupplierServiceID on this subscription
5,59e09f98-d1f1-1deb-6fa5-8cd3d73b8d7f,NaN,None,None,None,None,None,None,None,False,no SupplierServiceID on this subscription
6,doris.dai@vygr.net,1638359637,None,None,None,None,None,None,None,False,circuits lookup failed: 404 Client Error: Not ...
7,Discount for V112996269,NaN,None,None,None,None,None,None,None,False,no SupplierServiceID on this subscription
8,Discount for V113067904,NaN,None,None,None,None,None,None,None,False,no SupplierServiceID on this subscription
10,al-faris.ali@vygr.net,1637273661,None,None,None,None,None,None,None,False,circuits lookup failed: 404 Client Error: Not ...
11,21530367627SLC001695,NaN,None,None,None,None,None,None,None,False,no SupplierServiceID on this subscription


## 5. Save

In [12]:
save_df("staff_subscriptions_resolved", df_subscriptions)


Saved 210 rows -> migration_data\05_staff_subscriptions_resolved.csv
